In [15]:
# ---- MBE + GAPT ---- 
# 1. Modified GPT (with MBE calculation)
from src.gapt import GPTConfig, GPT
import torch 

config = GPTConfig(
    n_layer=4,
    n_head=4,
    n_embd=128,
)
    
model = GPT(config)
# model = model.to("cuda")
# model = torch.compile(model)

In [30]:
from src.model import GPTConfig as BaseGPTConfig
from src.model import GPT as BaseGPT

base_config = BaseGPTConfig(
    n_layer=4,
    n_head=4,
    n_embd=128,
)
    
base_model = BaseGPT(base_config)

In [53]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0
        
        buf = tokens[pos : pos + sequence_length + 1]
        inputs = buf[None, :-1].to(device=device, dtype=torch.int32, non_blocking=True) # no sync on host side;
        targets = buf[None, 1:].to(device=device, dtype=torch.int64, non_blocking=True) 
        pos += sequence_length
        yield inputs, targets

data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))
train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=256, device="cpu")

In [61]:
import time
from src.mbe import patch_mbe

# --- forward propagation time log ---
input, target = next(train_loader)

model.enable_timing = True
output = model(input, target, attn_blocksize=256, patch_size=32)
model.enable_timing = False

# --- backward propagation ---
backward_start = time.time()
loss = output['entropy'] + torch.stack([output[k] for k in output if k.startswith('mbe_')]).mean()
loss.backward()
backward_end = time.time()
backward_time = backward_end - backward_start
print(f"Backward time: {backward_time * 1000:.1f} ms")



⏱️  Forward Pass Timing Breakdown (ms)
Setup (mask + embed):         3.44 ms  (  7.0%)
Encoder Forward:             16.51 ms  ( 33.7%)
Encoder MBE:                  0.22 ms  (  0.4%)
Decoder Forward:             14.26 ms  ( 29.1%)
Decoder MBE:                  0.19 ms  (  0.4%)
Output (head + loss):        14.44 ms  ( 29.4%)
------------------------------------------------------------
TOTAL:                       49.06 ms

Backward time: 49.7 ms


In [58]:
import time
from src.mbe import patch_mbe

# --- forward propagation time log ---
input, target = next(train_loader)

foward_start = time.time()
loss = base_model(input, target, attn_blocksize=256)
foward_end = time.time()
foward_time = foward_end - foward_start
print(f"Forward time: {foward_time * 1000:.1f} ms")

# --- backward propagation ---
backward_start = time.time()
loss.backward()
backward_end = time.time()
backward_time = backward_end - backward_start
print(f"Backward time: {backward_time * 1000:.1f} ms")

Forward time: 55.6 ms
Backward time: 47.2 ms


In [6]:
# Toy training loop 
# -------------------------------------------------------------
attn_blocksize = 1792 
patch_size = 16 
no_reg = False 
use_gapt = True
log_grad_info = False
mbe_weight = 1.0

from src.utils import compute_loss
from src.gapt import GatedPhaseTransition
from src.gradtracker import GradientTracker, GradStatsRecorder, track_gradient_similarity
from torch.optim import Adam

grad_tracker = GradientTracker(model)
grad_stats = GradStatsRecorder()
gapt = GatedPhaseTransition()
iterations = 100 
optimizer = Adam(model.parameters(), lr=0.001)

inputs, targets = next(train_loader)

for i in range(iterations): 
    # inputs, targets = next(train_loader)
    loss_dict = model.forward(inputs, targets, attn_blocksize, patch_size)
    compute_loss(loss_dict)  

    # --- aggregate loss ---
    loss_dict = {
        "entropy": loss_dict["entropy"],
        "mbe": sum(v for k, v in loss_dict.items() if k.startswith("mbe_"))
    }
    stats = track_gradient_similarity(model, loss_dict["entropy"], loss_dict["mbe"]) 
    grad_stats.record(stats, i)
    loss_dict = {"entropy": loss_dict["entropy"]}

    # --- backward ---
    if log_grad_info: 
        grad_tracker.backward_with_tracking(loss_dict)
    else: 
        grad_tracker.backward(loss_dict)

    optimizer.step()
    optimizer.zero_grad()    

    # print(f" - step {i}/{iterations} " + "".join([f" {k}={v:.2f}" for k, v in loss_dict.items()]))
    print(f" - step {i}/{iterations} cosine similarity: {stats['global_cosine']:.2f}")
    # break

 - step 0/100 cosine similarity: 0.07
 - step 1/100 cosine similarity: 0.33
 - step 2/100 cosine similarity: 0.71
 - step 3/100 cosine similarity: 0.81
 - step 4/100 cosine similarity: 0.75
 - step 5/100 cosine similarity: 0.27
 - step 6/100 cosine similarity: -0.57
 - step 7/100 cosine similarity: -0.77
 - step 8/100 cosine similarity: -0.74
 - step 9/100 cosine similarity: -0.50
 - step 10/100 cosine similarity: 0.01
 - step 11/100 cosine similarity: 0.50
 - step 12/100 cosine similarity: 0.66
 - step 13/100 cosine similarity: 0.57
 - step 14/100 cosine similarity: 0.34
 - step 15/100 cosine similarity: 0.10
 - step 16/100 cosine similarity: -0.14
 - step 17/100 cosine similarity: -0.32
 - step 18/100 cosine similarity: -0.46
 - step 19/100 cosine similarity: -0.56
 - step 20/100 cosine similarity: -0.54
 - step 21/100 cosine similarity: -0.34
 - step 22/100 cosine similarity: 0.05
 - step 23/100 cosine similarity: 0.38
 - step 24/100 cosine similarity: 0.49
 - step 25/100 cosine sim

In [10]:
grad_stats.per_param_history['transformer.h.1.attn.c_k.weight']

[{'step': 0,
  'cosine': -0.015047072432935238,
  'ce_norm': 0.00017731536354403943,
  'mbe_norm': 0.014626403339207172},
 {'step': 1,
  'cosine': 0.3363814055919647,
  'ce_norm': 0.0005666902870871127,
  'mbe_norm': 0.021894410252571106},
 {'step': 2,
  'cosine': 0.654384195804596,
  'ce_norm': 0.0011747038224712014,
  'mbe_norm': 0.09413367509841919},
 {'step': 3,
  'cosine': 0.7339510321617126,
  'ce_norm': 0.0016687399474903941,
  'mbe_norm': 0.20515243709087372},
 {'step': 4,
  'cosine': 0.4775647521018982,
  'ce_norm': 0.0013239444233477116,
  'mbe_norm': 0.2548329532146454},
 {'step': 5,
  'cosine': -0.8291535973548889,
  'ce_norm': 0.0017008319264277816,
  'mbe_norm': 0.21904487907886505},
 {'step': 6,
  'cosine': -0.9615117311477661,
  'ce_norm': 0.0030312188901007175,
  'mbe_norm': 0.15622061491012573},
 {'step': 7,
  'cosine': -0.9583733677864075,
  'ce_norm': 0.00279521313495934,
  'mbe_norm': 0.11765563488006592},
 {'step': 8,
  'cosine': -0.8262987732887268,
  'ce_norm': 

In [11]:
grad_stats.save("temp_grad_stats.pkl")

Saved gradient stats to temp_grad_stats.pkl


In [ ]:
# ============================================================
# Computational Overhead Sweep: Base vs MBE
# ============================================================
import time
import torch
import pandas as pd

def get_peak_memory_mb():
    """Get peak memory in MB."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0

def benchmark_forward_backward(model, input_ids, targets, attn_blocksize, patch_size=None, 
                                n_warmup=3, n_runs=10, compute_mbe=False):
    """Benchmark forward + backward pass."""
    device = next(model.parameters()).device
    input_ids = input_ids.to(device)
    targets = targets.to(device)
    
    # Warmup
    for _ in range(n_warmup):
        model.zero_grad()
        if compute_mbe:
            out = model(input_ids, targets, attn_blocksize=attn_blocksize, patch_size=patch_size)
            loss = out['entropy'] + torch.stack([out[k] for k in out if k.startswith('mbe_')]).mean()
        else:
            loss = model(input_ids, targets, attn_blocksize=attn_blocksize)
        loss.backward()
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Timed runs
    times = []
    for _ in range(n_runs):
        model.zero_grad()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        
        start = time.perf_counter()
        if compute_mbe:
            out = model(input_ids, targets, attn_blocksize=attn_blocksize, patch_size=patch_size)
            loss = out['entropy'] + torch.stack([out[k] for k in out if k.startswith('mbe_')]).mean()
        else:
            loss = model(input_ids, targets, attn_blocksize=attn_blocksize)
        loss.backward()
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append((end - start) * 1000)  # ms
    
    peak_mem = get_peak_memory_mb()
    avg_time = sum(times) / len(times)
    std_time = (sum((t - avg_time)**2 for t in times) / len(times)) ** 0.5
    
    return {
        'time_ms': avg_time,
        'time_std': std_time,
        'peak_mem_mb': peak_mem,
        'tokens_per_sec': input_ids.numel() / (avg_time / 1000),
    }

# Sweep configs
SEQ_LENGTHS = [256, 512, 1024, 2048]
PATCH_SIZES = [8, 16, 32, 64]
ATTN_BLOCKSIZE = 256

results = []
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
base_model = base_model.to(device)

print(f"Device: {device}\n" + "=" * 70)

for seq_len in SEQ_LENGTHS:
    train_loader_tmp = data_generator(
        filename_pattern="data/fineweb10B/fineweb_train_*.bin",
        sequence_length=seq_len, device=device
    )
    input_ids, targets = next(train_loader_tmp)
    
    # Baseline
    base_stats = benchmark_forward_backward(
        base_model, input_ids, targets, 
        attn_blocksize=min(ATTN_BLOCKSIZE, seq_len), compute_mbe=False
    )
    results.append({'seq_len': seq_len, 'patch_size': '-', 'mode': 'Base', **base_stats})
    print(f"[seq={seq_len}] Base: {base_stats['time_ms']:.1f}ms | {base_stats['peak_mem_mb']:.0f}MB")
    
    # MBE sweep
    for ps in PATCH_SIZES:
        if ps > seq_len: continue
        mbe_stats = benchmark_forward_backward(
            model, input_ids, targets,
            attn_blocksize=min(ATTN_BLOCKSIZE, seq_len), patch_size=ps, compute_mbe=True
        )
        time_oh = (mbe_stats['time_ms'] / base_stats['time_ms'] - 1) * 100
        mem_oh = (mbe_stats['peak_mem_mb'] / base_stats['peak_mem_mb'] - 1) * 100 if base_stats['peak_mem_mb'] > 0 else 0
        results.append({
            'seq_len': seq_len, 'patch_size': ps, 'mode': 'MBE',
            **mbe_stats, 'time_oh_%': time_oh, 'mem_oh_%': mem_oh
        })
        print(f"[seq={seq_len}] MBE(ps={ps}): {mbe_stats['time_ms']:.1f}ms (+{time_oh:.1f}%) | {mbe_stats['peak_mem_mb']:.0f}MB (+{mem_oh:.1f}%)")

df = pd.DataFrame(results)
print("\n" + df.to_string(index=False))

Device: cpu
[seq=256] Base: 81.0ms | 0MB
[seq=256] MBE(ps=8): 80.3ms (+-0.9%) | 0MB (+0.0%)
[seq=256] MBE(ps=16): 80.5ms (+-0.6%) | 0MB (+0.0%)
[seq=256] MBE(ps=32): 90.5ms (+11.7%) | 0MB (+0.0%)
[seq=256] MBE(ps=64): 81.5ms (+0.6%) | 0MB (+0.0%)
[seq=512] Base: 143.9ms | 0MB
[seq=512] MBE(ps=8): 143.4ms (+-0.4%) | 0MB (+0.0%)
[seq=512] MBE(ps=16): 143.6ms (+-0.3%) | 0MB (+0.0%)
[seq=512] MBE(ps=32): 142.5ms (+-1.0%) | 0MB (+0.0%)
[seq=512] MBE(ps=64): 143.4ms (+-0.4%) | 0MB (+0.0%)
[seq=1024] Base: 248.2ms | 0MB
[seq=1024] MBE(ps=8): 253.2ms (+2.0%) | 0MB (+0.0%)
[seq=1024] MBE(ps=16): 251.7ms (+1.4%) | 0MB (+0.0%)
[seq=1024] MBE(ps=32): 252.0ms (+1.5%) | 0MB (+0.0%)
[seq=1024] MBE(ps=64): 253.6ms (+2.2%) | 0MB (+0.0%)
